# QDac Test Notebook

Instantiates `instruments.qdac.QDac`, connects, and exercises its channel
configuration, square-wave trigger/gate configuration, and error-check
methods.

**No real instrument is connected to this development machine.** Every
live-hardware call below is wrapped in `try`/`except` so this notebook
runs cleanly end-to-end and prints a clear "hardware not available"
message instead of raising, when run here. On the lab computer (with the
QDac actually connected), the same cells should run against real hardware.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from instruments.qdac import QDac, QDacError
from instruments import config

print(f"Default QDac VISA address: {config.QDAC_VISA_ADDRESS}")

## Connect

In [ ]:
qdac = QDac()
qdac_connected = False

try:
    qdac.connect()
    qdac_connected = True
    print(f"Connected to QDac at {qdac.resource}")
except Exception as exc:
    print(f"Hardware not available: could not connect to QDac ({exc})")

## Error queue

In [ ]:
try:
    if not qdac_connected:
        raise RuntimeError("QDac not connected")
    print("QDac error queue:", qdac.get_error())
    qdac.check_error()
    print("No pending QDac error.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Channel initialization and configuration

In [ ]:
try:
    if not qdac_connected:
        raise RuntimeError("QDac not connected")
    qdac.channel_init(1)
    # vrange must cover the square-wave span configured later (5 V, i.e.
    # 0-5 V) -- "Low" only covers 2 V, so "High" (10 V) is required here.
    qdac.channel_set(1, filter="Med", vrange="High", crange="High")
    print("Channel 1 initialized and configured.")
    print("Channel 1 range:", qdac.get_channel_range(1))
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

In [ ]:
try:
    if not qdac_connected:
        raise RuntimeError("QDac not connected")
    qdac.set_channel_voltage(1, 0.0)
    print("Channel 1 voltage set to 0.0 V.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

In [ ]:
try:
    if not qdac_connected:
        raise RuntimeError("QDac not connected")
    qdac.set_slew_rate_for_channels([1, 2], 2e6)
    print("Slew rate set for channels 1 and 2.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Trigger/gate square-wave configuration

In this experiment the QDac is used purely as a trigger/gate signal
generator that synchronizes the Avtech pulse generator and the MFLI
lock-in amplifier -- see `configure_square_wave` in `instruments/qdac.py`.

In [ ]:
try:
    if not qdac_connected:
        raise RuntimeError("QDac not connected")
    gate_voltage = 5.0
    qdac.configure_square_wave(
        1,
        frequency=200,
        span=gate_voltage,
        offset=gate_voltage / 2,
        trigger_source=1,
    )
    qdac.start_square_wave(1)
    qdac.fire_internal_trigger(1)
    print("Square-wave trigger/gate on channel 1 configured, armed, and fired.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

In [ ]:
try:
    if not qdac_connected:
        raise RuntimeError("QDac not connected")
    qdac.abort_square_wave(1)
    qdac.set_channel_voltage(1, 0.0)
    print("Channel 1 square wave aborted and voltage reset to 0 V.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Disconnect

In [ ]:
try:
    if qdac_connected:
        qdac.disconnect()
        print("Disconnected from QDac.")
    else:
        print("Skipping disconnect: QDac was never connected.")
except Exception as exc:
    print(f"Error during disconnect: {exc}")

## Context manager usage

`QDac` (via `VisaInstrument`) also supports use as a context manager, which
guarantees the connection is closed even if an error occurs.

In [ ]:
try:
    with QDac() as qdac_cm:
        print("QDac error queue:", qdac_cm.get_error())
except Exception as exc:
    print(f"Hardware not available: {exc}")